# Clase 11 — Pandas: manipulación de datos para IA

En la Clase 10 conocimos `Series`, `DataFrame` y cómo leer y explorar un dataset.  
Los datos reales casi nunca vienen listos para usar: tienen valores faltantes, columnas que hay que calcular y preguntas que requieren agrupar o combinar tablas. Eso es justo lo que un modelo de IA necesita antes de poder entrenarse (algo que vamos a retomar en las próximas clases, con ejemplos de clasificación).

**Objetivos de la clase:**
- Filtrar filas con condiciones booleanas.
- Crear columnas nuevas a partir de columnas existentes.
- Detectar y tratar valores faltantes con `isna()`, `dropna()`, `fillna()`.
- Agrupar datos y calcular agregaciones con `groupby`.
- Combinar dos tablas con `merge`.
- Ordenar datos y exportarlos con `sort_values` y `to_csv`.


---
## 0. El dataset de trabajo

Vamos a seguir usando el dataset de frutas, ahora un poco más grande y con datos faltantes, tal como aparecerían en la vida real.


In [1]:
import pandas as pd
import numpy as np

frutas = pd.DataFrame({
    "nombre": [
        "manzana roja", "manzana verde", "naranja de mesa", "banana",
        "pera", "kiwi", "mandarina", "limón", "uva", "sandía",
    ],
    "clase": [
        "manzana", "manzana", "naranja", "banana",
        "pera", "kiwi", "naranja", "naranja", "uva", "sandía",
    ],
    "color": [
        "rojo", "verde", "naranja", "amarillo",
        "verde", "verde", "naranja", "verde", "verde", "verde",
    ],
    "calorias": [95, 80, 62, 89, 57, np.nan, 53, 29, 69, np.nan],
    "precio_kg": [1200, 1100, 900, 800, 1000, np.nan, 950, 700, 1500, 600],
})

frutas


,nombre,clase,color,calorias,precio_kg
0,manzana roja,manzana,rojo,95.0,1200.0
1,manzana verde,manzana,verde,80.0,1100.0
2,naranja de mesa,naranja,naranja,62.0,900.0
3,banana,banana,amarillo,89.0,800.0
4,pera,pera,verde,57.0,1000.0
5,kiwi,kiwi,verde,NaN,NaN
6,mandarina,naranja,naranja,53.0,950.0
7,limón,naranja,verde,29.0,700.0
8,uva,uva,verde,69.0,1500.0
9,sandía,sandía,verde,NaN,600.0


---
## 1. Filtrar filas con condiciones booleanas

Igual que filtrábamos arrays de NumPy con `array[array >= 7]` (Clase 9), en pandas filtramos filas con `df[condicion]`.


In [2]:
frutas_caras = frutas[frutas["precio_kg"] > 1000]
frutas_verdes_caras = frutas[(frutas["color"] == "verde") & (frutas["precio_kg"] > 900)]

print("Frutas con precio > 1000:")
print(frutas_caras[["nombre", "precio_kg"]])
print()
print("Frutas verdes y con precio > 900:")
print(frutas_verdes_caras[["nombre", "color", "precio_kg"]])


Frutas con precio > 1000:
          nombre  precio_kg
0   manzana roja     1200.0
1  manzana verde     1100.0
8            uva     1500.0

Frutas verdes y con precio > 900:
          nombre  color  precio_kg
1  manzana verde  verde     1100.0
4           pera  verde     1000.0
8            uva  verde     1500.0


---
## 2. Columnas derivadas

Una columna nueva se calcula igual que una operación vectorizada de NumPy: se opera la columna completa y se asigna a un nombre nuevo.


In [3]:
frutas["precio_100g"] = frutas["precio_kg"] / 10
frutas["es_dulce"] = frutas["calorias"] > 60   # regla simple, a modo de ejemplo

frutas[["nombre", "precio_kg", "precio_100g", "calorias", "es_dulce"]]


,nombre,precio_kg,precio_100g,calorias,es_dulce
0,manzana roja,1200.0,120.0,95.0,True
1,manzana verde,1100.0,110.0,80.0,True
2,naranja de mesa,900.0,90.0,62.0,True
3,banana,800.0,80.0,89.0,True
4,pera,1000.0,100.0,57.0,False
5,kiwi,NaN,NaN,NaN,False
6,mandarina,950.0,95.0,53.0,False
7,limón,700.0,70.0,29.0,False
8,uva,1500.0,150.0,69.0,True
9,sandía,600.0,60.0,NaN,False


---
## 3. Valores faltantes: `isna()`, `dropna()`, `fillna()`

`calorias` y `precio_kg` tienen valores faltantes (`NaN`). Antes de usar estos datos para cualquier análisis o modelo, hay que decidir qué hacer con ellos:

| Método | Qué hace |
|---|---|
| `.isna()` | Marca qué valores son faltantes (`True`/`False`) |
| `.dropna()` | Elimina las filas con al menos un valor faltante |
| `.fillna(valor)` | Completa los faltantes con un valor (ej. el promedio) |


In [4]:
print("¿Dónde hay valores faltantes?")
print(frutas[["nombre", "calorias", "precio_kg"]].isna())
print()
print("Filas totales antes de limpiar:", len(frutas))

frutas_sin_nulos = frutas.dropna()
print("Filas después de dropna()      :", len(frutas_sin_nulos))

frutas_completadas = frutas.copy()
frutas_completadas["calorias"] = frutas_completadas["calorias"].fillna(
    frutas_completadas["calorias"].mean()
)
frutas_completadas["precio_kg"] = frutas_completadas["precio_kg"].fillna(
    frutas_completadas["precio_kg"].mean()
)
print()
print("Calorías después de fillna(promedio):")
print(frutas_completadas[["nombre", "calorias"]])


¿Dónde hay valores faltantes?
   nombre  calorias  precio_kg
0   False     False      False
1   False     False      False
2   False     False      False
3   False     False      False
4   False     False      False
5   False      True       True
6   False     False      False
7   False     False      False
8   False     False      False
9   False      True      False

Filas totales antes de limpiar: 10
Filas después de dropna()      : 8

Calorías después de fillna(promedio):
            nombre  calorias
0     manzana roja     95.00
1    manzana verde     80.00
2  naranja de mesa     62.00
3           banana     89.00
4             pera     57.00
5             kiwi     66.75
6        mandarina     53.00
7            limón     29.00
8              uva     69.00
9           sandía     66.75


---
## 4. Agrupar con `groupby`

`groupby` responde preguntas del tipo "¿cuánto en promedio, por grupo?". Agrupamos por `color` y calculamos el promedio de calorías y de precio para cada grupo.


In [5]:
promedio_por_color = frutas_completadas.groupby("color")[["calorias", "precio_kg"]].mean()
cantidad_por_color = frutas_completadas.groupby("color")["nombre"].count()

print("Promedio de calorías y precio por color:")
print(promedio_por_color.round(1))
print()
print("Cantidad de frutas por color:")
print(cantidad_por_color)


Promedio de calorías y precio por color:
          calorias  precio_kg
color                        
amarillo      89.0      800.0
naranja       57.5      925.0
rojo          95.0     1200.0
verde         61.4      978.7

Cantidad de frutas por color:
color
amarillo    1
naranja     2
rojo        1
verde       6
Name: nombre, dtype: int64


---
## 5. Combinar tablas con `merge`

Es muy común que la información esté repartida en más de una tabla. Acá tenemos una segunda tabla con el país de origen de cada `clase` de fruta. `merge` las combina usando una columna en común.


In [6]:
origenes = pd.DataFrame({
    "clase": ["manzana", "naranja", "banana", "pera", "kiwi", "uva", "sandía"],
    "origen": ["Argentina", "España", "Ecuador", "Argentina", "Nueva Zelanda", "Chile", "Argentina"],
})

frutas_con_origen = frutas_completadas.merge(origenes, on="clase", how="left")

frutas_con_origen[["nombre", "clase", "origen"]]


,nombre,clase,origen
0,manzana roja,manzana,Argentina
1,manzana verde,manzana,Argentina
2,naranja de mesa,naranja,España
3,banana,banana,Ecuador
4,pera,pera,Argentina
5,kiwi,kiwi,Nueva Zelanda
6,mandarina,naranja,España
7,limón,naranja,España
8,uva,uva,Chile
9,sandía,sandía,Argentina


---
## 6. Ordenar y exportar

`sort_values` ordena filas por una columna. `to_csv` guarda el resultado en disco, listo para la próxima clase.


In [7]:
from pathlib import Path
import tempfile

frutas_ordenadas = frutas_con_origen.sort_values("calorias", ascending=False)
print(frutas_ordenadas[["nombre", "calorias", "origen"]])

carpeta_salida = Path(tempfile.gettempdir()) / "modulo_3_clase_11"
carpeta_salida.mkdir(exist_ok=True)
ruta_csv = carpeta_salida / "frutas_limpias.csv"

frutas_ordenadas.to_csv(ruta_csv, index=False)
print()
print("Archivo guardado en:", ruta_csv)


            nombre  calorias         origen
0     manzana roja     95.00      Argentina
3           banana     89.00        Ecuador
1    manzana verde     80.00      Argentina
8              uva     69.00          Chile
5             kiwi     66.75  Nueva Zelanda
9           sandía     66.75      Argentina
2  naranja de mesa     62.00         España
4             pera     57.00      Argentina
6        mandarina     53.00         España
7            limón     29.00         España

Archivo guardado en: /var/folders/_6/7q7fqgyn2y71j9nx9jk7_6980000gn/T/modulo_3_clase_11/frutas_limpias.csv


---
## 📝 Actividad 1 — Limpiar y filtrar

**Consigna:**
- A partir de `frutas` (la tabla original, con nulos), crear `frutas_actividad` completando los `NaN` de `calorias` con el promedio y los de `precio_kg` con la mediana (`.median()`).
- Filtrar las frutas con `calorias` mayor a 60.


In [8]:
# ACTIVIDAD 1: limpiar y filtrar
frutas_actividad = frutas.copy()

# TODO: completar calorias con el promedio
frutas_actividad["calorias"] = frutas_actividad["calorias"].fillna(
    frutas_actividad["calorias"].mean()
)

# TODO: completar precio_kg con la mediana
frutas_actividad["precio_kg"] = frutas_actividad["precio_kg"].fillna(
    frutas_actividad["precio_kg"].median()
)

# TODO: filtrar calorias > 60
frutas_altas_calorias = frutas_actividad[frutas_actividad["calorias"] > 60]

print(frutas_altas_calorias[["nombre", "calorias", "precio_kg"]])


            nombre  calorias  precio_kg
0     manzana roja     95.00     1200.0
1    manzana verde     80.00     1100.0
2  naranja de mesa     62.00      900.0
3           banana     89.00      800.0
5             kiwi     66.75      950.0
8              uva     69.00     1500.0
9           sandía     66.75      600.0


---
### Actividad 2 — Agrupar y exportar

**Consigna:**
- A partir de `frutas_con_origen`, agrupar por `origen` y calcular el promedio de `precio_kg`.
- Ordenar el resultado del origen más caro al más barato.


In [9]:
# ACTIVIDAD 2: agrupar por origen y ordenar
# TODO: promedio de precio_kg por origen
precio_promedio_por_origen = frutas_con_origen.groupby("origen")["precio_kg"].mean()

# TODO: ordenar de mayor a menor precio
precio_promedio_por_origen = precio_promedio_por_origen.sort_values(ascending=False)

print(precio_promedio_por_origen.round(1))


origen
Chile            1500.0
Argentina         975.0
Nueva Zelanda     972.2
España            850.0
Ecuador           800.0
Name: precio_kg, dtype: float64


---
## ✅ Resumen

| Herramienta | Qué resuelve |
|---|---|
| `df[condicion]` | Filtrar filas |
| `df["nueva_col"] = ...` | Crear columnas derivadas |
| `.isna()`, `.dropna()`, `.fillna(...)` | Detectar y tratar valores faltantes |
| `df.groupby(col)[...].mean()` | Agregaciones por grupo |
| `df.merge(otra_df, on=...)` | Combinar tablas |
| `.sort_values(...)`, `.to_csv(...)` | Ordenar y exportar resultados |

**Lo que construiste hoy:**
- un dataset limpio, con columnas derivadas y sin valores faltantes,
- estadísticas agrupadas por color y por origen,
- un archivo CSV listo para usar en la próxima clase.

**Lo que viene:**
- **Clase 12**: vamos a usar pandas para armar tu primer clasificador, con gráficos y una matriz de confusión.
